# Конспект 5. Ошибки и исключения

**Ошибки случаются у всех** например, пользователь ввёл слово вместо числа, в данных пустое поле,
файл назван с опечаткой. Без обработки подобных ошибок, программа на первой же такой мелочи
завершает свое выполнение. Сегодня научимся перехватывать ошибки и решать, что делать с этим дальше.

## Данные конспекта

In [1]:
TEXTS = [
    "Учёные открыли новый вид бактерий в глубинах Тихого океана",
    "Сборная выиграла матч со счётом 3:1",
    "Новый вирус обнаружен учёными в лаборатории Новосибирска",
]

LABELS = ["наука", "спорт", "наука"]
SCORES = [5, 3, 4]

print("записей:", len(TEXTS))

записей: 3


---

# Часть I. Основы

# 1. Как выглядит ошибка

Когда что-то идёт не так, Python останавливает программу и выводит сообщение.
Читайте его **снизу вверх**:

- **последняя строка** - тип ошибки и её текст, например `IndexError: list index out of range`;
- **стрелка `---->`** - строка, на которой всё сломалось.

Такая ошибка называется **исключением**. Частые исключения:

| Исключение | Когда | Пример |
| --- | --- | --- |
| `ValueError` | значение неподходящее | `int('три')` |
| `TypeError` | тип неподходящий | `int(None)`, `'5' + 3` |
| `ZeroDivisionError` | деление на ноль | `10 / 0` |
| `IndexError` | нет такого индекса | `TEXTS[10]` |
| `KeyError` | нет такого ключа | `{'a': 1}['b']` |
| `FileNotFoundError` | нет файла | `open('nope.txt')` |

In [ ]:
# раскомментируйте по одной строке и посмотрите на последнюю строку сообщения:
# print(TEXTS[10])
# print(int('три'))
# print(SCORES[0] / 0)

---

# 2. Первый `try / except`

Опасный код помещают в `try`. Если там случилась ошибка указанного типа, Python не
останавливается, а выполняет блок `except`. Если ошибки нет, `except` пропускается.

In [2]:
for answer in ['2', 'два']:
    try:
        number = int(answer)
        print('число:', number)
    except ValueError:
        print('не число:', answer)

число: 2
не число: два


Как только в `try` случилась ошибка, остальные строки `try` **не выполняются**:
`print('число:', ...)` для `'два'` не сработал.

---

# 3. Несколько `except`

Разные ошибки - разные блоки кода. Python проверяет их **сверху вниз** и выполняет первый
подходящий. Ошибку, для которой блока нет, `try` не спасёт и программа просто завершает свое выполнение.

In [3]:
for answer in ['1', 'один', '7']:
    try:
        print(answer, '->', TEXTS[int(answer)])
    except ValueError:
        print(answer, '-> не число')
    except IndexError:
        print(answer, '-> нет записи с таким номером')

1 -> Сборная выиграла матч со счётом 3:1
один -> не число
7 -> нет записи с таким номером


---

# 4. `as error`, `else`, `finally`

- `except ValueError as error` - ошибка сохраняется в переменную. `print(error)` покажет
  её текст, `type(error).__name__` - название типа.
- `except (ValueError, IndexError)` - один блок на несколько типов (в скобках).
- `else` - выполняется, если ошибки **не было**. В `try` оставляют только опасную строку.
- `finally` - выполняется **всегда**, была ошибка или нет.

| Что случилось в `try` | Что выполнится |
| --- | --- |
| ошибки нет | `else`, затем `finally` |
| ошибка, есть подходящий `except` | `except`, затем `finally` |
| ошибка, подходящего `except` нет | `finally`, затем программа остановится |

In [4]:
for answer in ['2', 'два', '5']:
    try:
        label = LABELS[int(answer)]
    except (ValueError, IndexError) as error:
        print('ошибка:', type(error).__name__, '-', error)
    else:
        print('метка:', label)
    finally:
        print('проверено:', answer)

метка: наука
проверено: 2
ошибка: ValueError - invalid literal for int() with base 10: 'два'
проверено: два
ошибка: IndexError - list index out of range
проверено: 5


---

# 5. `KeyError` и `get`

Если ключ в словаре отсуствует нужно использоват исключение `KeyError`. Его можно перехватить, а можно не допустить:
`d.get(ключ, 0)` вернёт `0`, если ключа нет. Если хватает простой проверки, `try` не нужен.

In [5]:
counts = {}
for label in LABELS:
    counts[label] = counts.get(label, 0) + 1
print(counts)

for label in ['наука', 'культура']:
    try:
        print(label, counts[label])
    except KeyError:
        print('метки', label, 'нет')

print('культура через get:', counts.get('культура', 0))

{'наука': 2, 'спорт': 1}
наука 2
метки культура нет
культура через get: 0


---

# Часть II. Ошибки в данных

# 6. Пропускаем плохую запись

В настоящих данных всегда есть неподходящие данные - пустые строки, слова вместо чисел, `None`.
Одна такая плохая запись не должна ломать обработку остальных. Поэтому `try` ставят
**внутри** цикла и плохое значение пропускается, остальные данные обрабатываем.

`int('')` - это `ValueError`, а `int(None)` - `TypeError`. Перехватываем оба.

In [6]:
raw_scores = ['5', '', '3', 'четыре', None, '4']

scores = []
skipped = 0
for value in raw_scores:
    try:
        scores.append(int(value))
    except (ValueError, TypeError):
        print('пропуск:', repr(value))
        skipped += 1

print('оценки:', scores, '| пропущено:', skipped)
print('средняя:', sum(scores) / len(scores))

пропуск: ''
пропуск: 'четыре'
пропуск: None
оценки: [5, 3, 4] | пропущено: 3
средняя: 4.0


> Если поставить `try` **снаружи** цикла, первая же плохая запись прервёт весь цикл -
> и остальные хорошие записи не обработаются. И никогда не пишите `except: pass`, в таком случае
> ошибки не выведутся, и Вы не узнаете, что данные потеряны.

---

# 7. `raise` - своя проверка

Функция может сама сообщить об ошибке: `raise ValueError('текст')`. Так она говорит
«с такими данными я работать не буду». Ошибка «всплывает» из функции там, где её
вызвали - там её и перехватывают.

In [7]:
def average(values):
    """Средняя оценка. Пустой список - ошибка."""
    if len(values) == 0:
        raise ValueError('нет оценок')
    return sum(values) / len(values)

for values in [SCORES, []]:
    try:
        print('средняя:', average(values))
    except ValueError as error:
        print('ошибка:', error)

средняя: 4.0
ошибка: нет оценок


Без `raise` функция упала бы на `sum([]) / len([])` с непонятным `ZeroDivisionError`.
Своё сообщение `нет оценок` объясняет, что именно не так.

---

# 8. Файла нет: `FileNotFoundError`

Чтение удобно спрятать в функцию и если файла нет, она сообщает об этом и возвращает
«пустой» результат, а программа продолжает работу.

In [ ]:
with open('texts.txt', 'w', encoding='utf-8') as file:
    for text in TEXTS:
        file.write(text + '\n')

def read_lines(name):
    """Строки файла или пустой список, если файла нет."""
    try:
        with open(name, 'r', encoding='utf-8') as file:
            return file.readlines()
    except FileNotFoundError:
        print('нет файла:', name)
        return []

print('texts.txt:', len(read_lines('texts.txt')), 'строк')
print('text.txt:', len(read_lines('text.txt')), 'строк')

---

# 9. Всё вместе: испорченный CSV и журнал ошибок

Настоящая обработка данных: создаём CSV, где часть строк испорчена, читаем его,
хорошие строки считаем, а плохие **записываем в журнал** `errors.txt` с номером строки
и текстом ошибки. Так ничего не теряется молча, и потом видно, что пришлось пропустить.

Обе ошибки здесь - `ValueError`: и `int('пять')`, и строка без запятой
(`a, b = 'спорт'.split(',')` - «not enough values to unpack»).

In [8]:
with open('scores.csv', 'w', encoding='utf-8') as file:
    file.write('label,score\n')
    for line in ['наука,5', 'спорт,пять', 'наука,4', 'спорт', 'спорт,3']:
        file.write(line + '\n')

with open('scores.csv', 'r', encoding='utf-8') as file:
    lines = file.readlines()

totals = {}
done = 0
with open('errors.txt', 'w', encoding='utf-8') as log:
    for number, line in enumerate(lines[1:], start=2):    # строка 1 - заголовок
        try:
            label, score = line.strip().split(',')
            score = int(score)
        except ValueError as error:
            log.write(f'строка {number}: {line.strip()} - {error}\n')
        else:
            totals[label] = totals.get(label, 0) + score
            done += 1

print('обработано:', done, '| пропущено:', len(lines) - 1 - done)
print('итоги:', totals)

with open('errors.txt', 'r', encoding='utf-8') as log:
    print(log.read())

обработано: 3 | пропущено: 2
итоги: {'наука': 9, 'спорт': 3}
строка 3: спорт,пять - invalid literal for int() with base 10: 'пять'
строка 5: спорт - not enough values to unpack (expected 2, got 1)



---

## Шпаргалка

| Что нужно | Как пишется | Важно |
| --- | --- | --- |
| перехватить ошибку | `try:` ... `except ValueError:` ... | ловите ожидаемый тип |
| несколько ошибок | несколько `except` подряд | сверху вниз, первый подходящий |
| несколько типов одним блоком | `except (ValueError, IndexError):` | в скобках |
| текст ошибки | `except ValueError as error: print(error)` | |
| тип ошибки | `type(error).__name__` | |
| если ошибки не было | `else:` | в `try` - только опасная строка |
| выполнить всегда | `finally:` | |
| ключа может не быть | `d.get(key, 0)` | без `try` |
| пропустить плохую запись | `try` **внутри** цикла | считайте пропущенные |
| сообщить об ошибке | `raise ValueError('текст')` | |
| нет файла | `except FileNotFoundError:` | |
| журнал ошибок | `log.write(f'строка {number}: {line.strip()} - {error}\n')` | |
| нельзя | `except: pass` | ошибки пропадут молча |

---

# Найди ошибку

Раскомментируйте, запустите и найдите причину. Ответы - после ячеек.

In [ ]:
# Ошибка 1
# try:
#     print(TEXTS[5])
# except ValueError:
#     print('нет такой записи')

In [ ]:
# Ошибка 2 - ошибки нет, но учтено только две оценки
# raw = ['5', '4', 'три', '3']
# scores = []
# try:
#     for value in raw:
#         scores.append(int(value))
# except ValueError:
#     print('плохое значение')
# print(scores)

In [ ]:
# Ошибка 3
# try:
#     chosen = int('два')
# except ValueError:
#     print('не число')
# print(LABELS[chosen])

In [ ]:
# Ошибка 4 - ошибки нет, но проверка не сработала
# def average(values):
#     if len(values) == 0:
#         print('нет оценок')
#     return sum(values) / len(values)
#
# try:
#     average([])
# except ValueError as error:
#     print('ошибка:', error)

In [ ]:
# Ошибка 5
# for value in ['5', None, '3']:
#     try:
#         print(int(value))
#     except ValueError:
#         print('пропуск:', repr(value))

**Ответы:**

1. `IndexError: list index out of range`. Номер за пределами списка - это `IndexError`,
   а не `ValueError`. Нужно `except IndexError`.
2. `try` стоит снаружи цикла: на `'три'` цикл прервался, и `'3'` уже не обработана.
   Нужно поставить `try` внутрь цикла.
3. `NameError: name 'chosen' is not defined`. `int('два')` упал, присваивание не
   выполнилось. Код, которому нужен результат `try`, пишут в `else`.
4. `print` не останавливает функцию: она дошла до деления и упала с
   `ZeroDivisionError`, которого `except ValueError` не ловит. Нужно
   `raise ValueError('нет оценок')`.
5. `TypeError: int() argument must be ... not 'NoneType'`. `int(None)` - это `TypeError`,
   а не `ValueError`. Нужно `except (ValueError, TypeError)`.